In [30]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [32]:
results_1l = pd.read_excel("resultados-1l.xlsx")
results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

results = pd.concat(
    [results_1l, results_2l],
    ignore_index=True
 )
#results = results_2l

In [33]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,MSE_ZZx1_theta,R2_ZZx2_theta,MSE_ZZx2_theta,...,R2_LSG_1_theta,MSE_LSG_1_theta,R2_LSG_2_theta,MSE_LSG_2_theta,R2_ZZx1_inv_theta,MSE_ZZx1_inv_theta,R2_zzx2_inv2_theta,MSE_zzx2_inv2_theta,R2_semiCirc_theta,MSE_semiCirc_theta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed4705,[1],0.3,0.7,0.01,4705,-0.179891,-0.000075,0.023074,-0.000823,...,-11.781966,-0.055668,-0.748364,-0.010603,-1.596818,-0.026972,-0.962770,-0.002709,-0.060649,-0.009015
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed2693,[1],0.3,0.7,0.01,2693,-0.214286,0.006568,0.015715,0.000107,...,-11.893951,-0.050805,-0.816748,-0.005875,-1.685503,-0.018900,-1.022080,-0.000460,-0.071675,-0.007226
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed4649,[1],0.3,0.7,0.01,4649,-0.210807,0.000878,0.012015,-0.000106,...,-11.975781,-0.056081,-0.808379,-0.010579,-1.689256,-0.027179,-1.022090,-0.002587,-0.077336,-0.009393
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed3633,[1],0.3,0.7,0.01,3633,0.761277,0.677203,-0.823908,0.462480,...,0.854605,0.595732,-4.782161,0.319042,0.018808,0.449090,-7.153187,0.079361,-30.210797,-0.196762
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed7789,[1],0.3,0.7,0.01,7789,-0.205858,0.002878,0.013170,0.001330,...,-11.954769,-0.054811,-0.796903,-0.009480,-1.683535,-0.025292,-1.038555,-0.001855,-0.078327,-0.010961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3086,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed6274,"[3, 1, 1]",0.7,0.3,0.90,6274,0.547843,0.582458,0.787891,0.362234,...,-2.400522,0.442405,-2.617589,0.255157,-2.139094,0.276627,-8.083965,0.142178,-10.509687,0.094532
3087,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed7805,"[3, 1, 1]",0.7,0.3,0.90,7805,0.524860,0.498866,0.635631,0.338082,...,-4.283150,0.347733,-2.038816,0.197777,-1.616691,0.271634,-5.912504,0.183614,-5.586486,0.102204
3088,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed3109,"[3, 1, 1]",0.7,0.3,0.90,3109,0.561345,0.517563,0.620186,0.353962,...,-3.522649,0.374510,-2.326809,0.224695,-0.493140,0.294403,-4.654585,0.193294,-7.037452,0.096106
3089,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed8496,"[3, 1, 1]",0.7,0.3,0.90,8496,0.560286,0.572841,-0.306579,0.350755,...,-2.391176,0.415491,-2.747356,0.223955,-0.831566,0.334386,-6.818385,0.161922,-8.428751,0.083885


In [34]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.0
w_train = 1.0
w_test = 0.0

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
       # - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
2762,model_arch77_r0.01_Ld0.5_Lp0.5_seed686,[77],0.975447,0.750666,-9.011131,0.975447
614,model_arch31_r0.01_Ld0.7_Lp0.3_seed7789,[31],0.970951,0.870563,-10.614379,0.970951
835,model_arch42_r0.9_Ld0.7_Lp0.3_seed4705,[42],0.970841,0.581076,-8.563372,0.970841
1956,model_arch98_r0.9_Ld0.7_Lp0.3_seed2693,[98],0.970461,0.841045,-9.862664,0.970461
1731,model_arch87_r0.01_Ld0.7_Lp0.3_seed2693,[87],0.970374,0.839846,-10.420557,0.970374



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZxReto_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
2762,model_arch77_r0.01_Ld0.5_Lp0.5_seed686,[77],0.975447,0.750666,0.294348,-27.277607,-15.300691,-1.471622,-3.726273,-3.418757,-12.177314,0.975447,0.750666,-9.011131,0.975447
614,model_arch31_r0.01_Ld0.7_Lp0.3_seed7789,[31],0.970951,0.870563,0.899175,-13.730204,-18.053348,0.793139,-2.738174,-4.649696,-36.821544,0.970951,0.870563,-10.614379,0.970951
835,model_arch42_r0.9_Ld0.7_Lp0.3_seed4705,[42],0.970841,0.581076,0.863056,-18.571110,-10.581124,0.698862,-3.560675,-2.147538,-26.645077,0.970841,0.581076,-8.563372,0.970841
1956,model_arch98_r0.9_Ld0.7_Lp0.3_seed2693,[98],0.970461,0.841045,0.904761,-20.036680,-18.938983,0.419342,-3.438624,-4.019647,-23.928817,0.970461,0.841045,-9.862664,0.970461
1731,model_arch87_r0.01_Ld0.7_Lp0.3_seed2693,[87],0.970374,0.839846,0.917063,-17.478285,-20.344416,0.676367,-3.344140,-4.982437,-28.388048,0.970374,0.839846,-10.420557,0.970374


In [35]:
final_table.to_excel("BestModels-otm.xlsx")